In [7]:
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# FDA API endpoint
url = "https://api.fda.gov/food/enforcement.json?limit=1000"

response = requests.get(url)
data = response.json()["results"]

# Convert to DataFrame
df = pd.DataFrame(data)

# Convert date
df["recall_initiation_date"] = pd.to_datetime(
    df["recall_initiation_date"],
    format="%Y%m%d",
    errors="coerce"
)

# --------------------------------------------------
# 1. Recall Classification Distribution
# --------------------------------------------------
class_counts = df["classification"].value_counts().reset_index()
class_counts.columns = ["Classification", "Count"]

fig1 = px.bar(
    class_counts,
    x="Classification",
    y="Count",
    title="FDA Food Recall Classifications",
    color="Classification"
)

fig1.show()

# --------------------------------------------------
# 2. Top Recall Reasons
# --------------------------------------------------
reasons = (
    df["reason_for_recall"]
    .value_counts()
    .head(10)
    .reset_index()
)

reasons.columns = ["Reason", "Count"]

fig2 = px.bar(
    reasons,
    x="Count",
    y="Reason",
    orientation="h",
    title="Top 10 Recall Reasons"
)

fig2.show()

# --------------------------------------------------
# 3. Recall Trend Over Time
# --------------------------------------------------
monthly = (
    df.groupby(df["recall_initiation_date"].dt.to_period("M"))
    .size()
    .reset_index(name="Count")
)

monthly["recall_initiation_date"] = monthly["recall_initiation_date"].astype(str)

fig3 = px.line(
    monthly,
    x="recall_initiation_date",
    y="Count",
    title="Food Recall Trend Over Time"
)

fig3.show()

# --------------------------------------------------
# 4. Top Recalling Firms
# --------------------------------------------------
firms = (
    df["recalling_firm"]
    .value_counts()
    .head(10)
    .reset_index()
)

firms.columns = ["Firm", "Count"]

fig4 = px.bar(
    firms,
    x="Count",
    y="Firm",
    orientation="h",
    title="Top 10 Recalling Firms"
)

fig4.show()

#Data_loader.py

In [8]:
import requests
import pandas as pd

def load_fda_data(limit=1000):
    url = f"https://api.fda.gov/food/enforcement.json?limit={limit}"

    response = requests.get(url)
    data = response.json()["results"]

    return pd.DataFrame(data)

#Anaylsis.py

In [9]:
def classify_priority(classification):

    if classification == "Class I":
        return "Critical"

    elif classification == "Class II":
        return "Medium"

    return "Low"

#app.py

In [10]:
import streamlit as st
import pandas as pd
import plotly.express as px

# load_fda_data is defined in the earlier data-loader cell.
# classify_priority is defined in the earlier analysis cell.

st.title("FDA Food Recall Dashboard")

df = load_fda_data()

df["Priority"] = df["classification"].apply(classify_priority)

# Recall Classifications
st.subheader("Recall Classification")

fig = px.bar(
    df["classification"].value_counts().reset_index(),
    x="classification",
    y="count"
)

st.plotly_chart(fig)

# Priority Summary
st.subheader("Priority Distribution")

fig2 = px.pie(
    df,
    names="Priority"
)

st.plotly_chart(fig2)

2026-09-01 16:12:34.416 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:35.835 
  command:

    streamlit run c:\Users\Monic\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-09-01 16:12:35.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:35.874 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:39.987 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:39.997 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:40.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:

DeltaGenerator()

In [11]:
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Overview",
    "Trends",
    "Contamination Types",
    "Geographic Impact",
    "Firm Analysis",
])


2026-09-01 16:12:41.220 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:41.224 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:41.228 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:41.232 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:41.236 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:12:41.240 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [13]:
import streamlit as st
import pandas as pd
import plotly.express as px
# load_fda_data is defined in the earlier data-loader cell.
from analysis import classify_priority

st.set_page_config(
    page_title="FDA Food Recall Dashboard",
    page_icon="🧪",
    layout="wide",
)

st.title("FDA Food Recall Dashboard")

df = load_fda_data()
if df.empty:
    st.warning("No recall data was returned from the FDA API.")
    st.stop()

df["Priority"] = df["classification"].apply(classify_priority)

# Overview metrics
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Recalls", len(df))
col2.metric("Critical Recalls", int((df["classification"] == "Class I").sum()))
col3.metric("Ongoing Recalls", int((df["status"] == "Ongoing").sum()))
col4.metric("Affected Firms", int(df["recalling_firm"].nunique()))

st.subheader("Recall Classification")
classification_chart = px.pie(df, names="classification", title="Recall Class Distribution")
st.plotly_chart(classification_chart, use_container_width=True)

# Tabs for additional views
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "Overview",
    "Trends",
    "Contamination Types",
    "Geographic Impact",
    "Firm Analysis",
])


2026-09-01 16:14:48.860 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:14:48.866 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:14:48.869 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:14:48.872 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:14:52.521 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:14:52.525 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:14:52.568 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:14:52.573 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [14]:
with tab1:

    st.header("Dashboard Overview")

    total_recalls = len(df)

    critical = len(
        df[df["classification"] == "Class I"]
    )

    ongoing = len(
        df[df["status"] == "Ongoing"]
    )

    firms = df["recalling_firm"].nunique()

    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Total Recalls", total_recalls)
    col2.metric("Critical Recalls", critical)
    col3.metric("Ongoing Recalls", ongoing)
    col4.metric("Affected Firms", firms)

    classification_chart = px.pie(
        df,
        names="classification",
        title="Recall Class Distribution"
    )

    st.plotly_chart(
        classification_chart,
        use_container_width=True
    )

2026-09-01 16:15:11.176 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:11.178 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:11.183 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:11.195 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:11.199 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:11.201 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:11.203 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:11.207 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [17]:
with tab2:

    st.header("Recall Trends")

    df["recall_initiation_date"] = pd.to_datetime(
        df["recall_initiation_date"],
        format="%Y%m%d",
        errors="coerce"
    )

    monthly_trend = (
        df.dropna(subset=["recall_initiation_date"])
        .groupby(df["recall_initiation_date"].dt.to_period("M"))
        .size()
        .reset_index(name="Count")
    )

    monthly_trend["Month"] = (
        monthly_trend["recall_initiation_date"]
        .astype(str)
    )

    trend_chart = px.line(
        monthly_trend,
        x="Month",
        y="Count",
        markers=True,
        title="Monthly Recall Trend"
    )

    st.plotly_chart(
        trend_chart,
        use_container_width=True
    )

2026-09-01 16:16:36.105 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:16:36.110 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:16:36.112 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:16:36.477 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 16:16:36.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:16:36.489 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:16:36.492 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [16]:
with tab3:

    st.header("Contamination Types")

    top_reasons = (
        df["reason_for_recall"]
        .value_counts()
        .head(15)
        .reset_index()
    )

    top_reasons.columns = [
        "Reason",
        "Count"
    ]

    reason_chart = px.bar(
        top_reasons,
        x="Count",
        y="Reason",
        orientation="h",
        title="Top Recall Causes"
    )

    st.plotly_chart(
        reason_chart,
        use_container_width=True
    )

    st.dataframe(top_reasons)

2026-09-01 16:15:34.260 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:34.262 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:34.269 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:34.680 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 16:15:34.692 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:34.694 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:15:34.700 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


# FDA Food Recall Analytics Dashboard

Real-world food recall monitoring project using
the FDA Food Enforcement Dataset.

## Run

streamlit run app.py

In [18]:
with tab4:

    st.header("Geographic Impact")

    state_counts = (
        df["state"]
        .value_counts()
        .head(20)
        .reset_index()
    )

    state_counts.columns = [
        "State",
        "Recalls"
    ]

    geo_chart = px.bar(
        state_counts,
        x="State",
        y="Recalls",
        color="Recalls",
        title="Top States by Recall Count"
    )

    st.plotly_chart(
        geo_chart,
        use_container_width=True
    )

    st.dataframe(state_counts)

2026-09-01 16:18:09.356 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:18:09.359 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:18:09.361 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:18:09.646 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 16:18:09.702 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:18:09.714 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:18:09.722 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [20]:
with tab4:

    st.header("🗺 Geographic Impact")

    # Count recalls per state
    state_counts = (
        df["state"]
        .value_counts()
        .reset_index()
    )

    state_counts.columns = [
        "state",
        "recall_count"
    ]

    # US Choropleth Map
    map_fig = px.choropleth(
        state_counts,
        locations="state",
        locationmode="USA-states",
        color="recall_count",
        scope="usa",
        color_continuous_scale="Reds",
        hover_name="state",
        hover_data=["recall_count"],
        title="Food Recalls by State"
    )

    map_fig.update_layout(
        height=600,
        margin=dict(l=0, r=0, t=50, b=0)
    )

    st.plotly_chart(
        map_fig,
        use_container_width=True
    )

    st.subheader("Top 20 States")

    top_states = (
        state_counts
        .sort_values(
            "recall_count",
            ascending=False
        )
        .head(20)
    )

    st.dataframe(
        top_states,
        use_container_width=True
    )

2026-09-01 16:20:24.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:24.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:24.543 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:25.497 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 16:20:25.506 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:25.510 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:25.512 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [21]:
selected_states = st.sidebar.multiselect(
    "Select States",
    sorted(df["state"].dropna().unique())
)

if selected_states:
    df = df[df["state"].isin(selected_states)]

2026-09-01 16:20:59.824 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:59.826 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:59.836 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:59.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:20:59.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [25]:
df["Year"] = df["recall_initiation_date"].dt.year
print()

In [30]:
# Sidebar Filters
st.sidebar.header("Filters")

# Year Dropdown
years = sorted(
    df["Year"].dropna().unique(),
    reverse=True
)

selected_year = st.sidebar.selectbox(
    "Select Year",
    options=["All Years"] + list(years)
)

# Apply filter
if selected_year != "All Years":
    df = df[df["Year"] == selected_year]

print()

2026-09-01 16:27:44.287 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:27:44.292 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:27:44.296 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:27:44.300 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:27:44.304 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:27:44.310 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:27:44.314 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:27:44.320 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [28]:
selected_years = st.sidebar.multiselect(
    "Select Year(s)",
    options=sorted(df["Year"].dropna().unique()),
    default=sorted(df["Year"].dropna().unique())[-3:]
)

if selected_years:
    df = df[df["Year"].isin(selected_years)]
    
print(df)

2026-09-01 16:26:22.035 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:26:22.039 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:26:22.042 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:26:22.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:26:22.046 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


         status         city state        country classification openfda  \
14   Terminated   Sioux City    IA  United States        Class I      {}   
17      Ongoing     San Luis    AZ  United States        Class I      {}   
21   Terminated      Detroit    MI  United States       Class II      {}   
25   Terminated  Lake Forest    IL  United States       Class II      {}   
31      Ongoing       Rogers    MN  United States       Class II      {}   
..          ...          ...   ...            ...            ...     ...   
982     Ongoing    Rochester    NY  United States       Class II      {}   
987  Terminated   Brownsburg    IN  United States       Class II      {}   
992  Terminated   Lake Worth    FL  United States        Class I      {}   
994  Terminated      Chicago    IL  United States        Class I      {}   
996  Terminated   Sioux City    IA  United States        Class I      {}   

    product_type event_id              recalling_firm            address_1  \
14       

In [35]:
if selected_year == "All Years":
    st.info("Viewing all available FDA recall records")
else:
    st.info(f"Viewing FDA recall records for {selected_year}")
    print(info)

2026-09-01 16:29:39.281 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:29:39.285 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:29:39.291 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [37]:
st.sidebar.header("Dashboard Filters")

# Year Filter
selected_year = st.sidebar.selectbox(
    "Year",
    options=["All Years"] + sorted(df["Year"].dropna().unique(), reverse=True),
)

if selected_year != "All Years":
    df = df[df["Year"] == selected_year]

# Recall Class Filter
selected_class = st.sidebar.multiselect(
    "Recall Classification",
    options=df["classification"].unique(),
    default=df["classification"].unique()
)

# Priority Filter
selected_priority = st.sidebar.multiselect(
    "Priority",
    options=df["Priority"].unique(),
    default=df["Priority"].unique()
)

# Apply filters
df = df[df["classification"].isin(selected_class)]
df = df[df["Priority"].isin(selected_priority)]

2026-09-01 16:31:09.570 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:31:09.574 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:31:09.578 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:31:09.587 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:31:09.591 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:31:09.595 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:31:09.597 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:31:09.599 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [38]:
# Product Search
product_search = st.sidebar.text_input(
    "🔍 Search Product Name",
    placeholder="Enter product name..."
)

2026-09-01 16:32:29.198 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:32:29.206 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:32:29.208 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:32:29.213 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:32:29.215 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:32:29.239 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [46]:
# Product Name Filter
if product_search:
    df = df[
        df["product_description"]
        .fillna("")
        .str.contains(
            product_search,
            case=False,
            na=False,
            regex=False,
        )
    ]

In [42]:
if product_search:
    matches = len(df)

    st.sidebar.success(
        f"{matches} matching recall(s)"
    )

In [43]:
st.sidebar.header("Dashboard Filters")

# Year
selected_year = st.sidebar.selectbox(
    "Year",
    ["All Years"] + list(years)
)

# Product Search
product_search = st.sidebar.text_input(
    "🔍 Product Search"
)

# Recall Classification
selected_class = st.sidebar.multiselect(
    "Classification",
    options=df["classification"].unique(),
    default=df["classification"].unique()
)

# Priority
selected_priority = st.sidebar.multiselect(
    "Priority",
    options=df["Priority"].unique(),
    default=df["Priority"].unique()
)

2026-09-01 16:48:51.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:51.835 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:51.842 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:51.847 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:51.849 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:51.851 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:51.857 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:51.859 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [44]:
st.subheader("Critical Recall Search Results")

st.dataframe(
    critical_df[
        [
            "product_description",
            "recalling_firm",
            "reason_for_recall",
            "status"
        ]
    ],
    use_container_width=True
)

2026-09-01 16:48:58.166 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:58.169 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:58.173 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:58.195 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-09-01 16:48:58.642 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:58.655 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-01 16:48:58.667 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()